<a href="https://colab.research.google.com/github/ktodome/b3exp/blob/master/notebooks/alpha_ray_day4_prime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# day4-prime — ローカルCSVの統合とアップロード

このノートは単独でGoogle Colab / Jupyterで実行できます。下の2つのコードセルを上から1回ずつ実行してください（`requests`・`ipywidgets`が必要です。Colabには標準で用意されています）。

1. day1・day2と同じIDとPINで「ログイン / 登録」を押します。既存の認証先を使用するため、未登録のIDは登録されます。
2. 同じ設定で測定したローカルCSVを2つ以上、一度に選択します。
3. 全ファイル共通のData Typeと測定条件（材質・角度・スリット・距離など）を入力し、同じ設定で測定したことを確認してチェックします。CSV自体にはこれらの条件が含まれないため、ノートが自動で条件の一致を判定することはできません。
4. 「統合内容を確認」で保存先のID、各測定時間、合計時間、総カウントを確認してから「統合結果をアップロード」を押します。

通信はログインと統合結果のアップロードのみです。スプレッドシートの保存済みデータは取得しません。統合ファイル1件をログインしたIDで追加します。送信完了の表示は既存APIの成功応答に基づきます。

In [1]:
# @title 1. ファイル形式と統合処理の準備
import base64
import csv
import hashlib
import io
import json
import math
from dataclasses import dataclass
from datetime import datetime, timedelta
from decimal import Decimal, InvalidOperation

CHANNELS = 1024
GAS_URL = 'https://script.google.com/macros/s/AKfycbzqAFZmMLfyITKDOtJSLw36D-ZvdO9NHIHIX19vsGOQxVJlB6CzRMhobYFTjFPKpmGi_g/exec'
SETTING_FIELDS = {
    'Pulse Height': ('voltage', 'channel'),
    'Source': ('channel', 'slit2', 'distance'),
    'Thickness': ('material',),
    'Measurement': ('material', 'angle', 'slit1', 'slit2', 'distance'),
}


@dataclass(frozen=True)
class Spectrum:
    name: str
    counts: tuple
    start: datetime
    end: datetime
    duration_sec: int
    original_time: Decimal
    original_flag: int


def numeric_cell(value, label, positive=False):
    text = str(value).strip().strip('"').strip()
    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError(f'{label}: 数値ではありません。') from exc
    if not number.is_finite() or number < 0 or (positive and number == 0):
        raise ValueError(f'{label}: 有効な{"正" if positive else "非負"}の数値が必要です。')
    return number


def parse_spectrum(content, name='file.csv'):
    """Read exactly 1024 channels and six metadata rows; never guess missing times."""
    if isinstance(content, (bytes, bytearray, memoryview)):
        data = bytes(content)
        try:
            content = data.decode('utf-8-sig')
        except UnicodeDecodeError:
            content = data.decode('cp932')
    if isinstance(content, str):
        try:
            rows = list(csv.reader(io.StringIO(content.lstrip('\ufeff').rstrip('\r\n')), strict=True))
        except csv.Error as exc:
            raise ValueError(f'{name}: CSVの形式が不正です。') from exc
        if any(len(row) != 1 for row in rows):
            raise ValueError(f'{name}: 1行1列のCSVが必要です（空行も不可）。')
        values = [row[0] for row in rows]
    else:
        # A complete column can also be checked offline against the supplied workbook.
        values = list(content)
    if len(values) != CHANNELS + 6:
        raise ValueError(f'{name}: 1030行が必要です。現在は{len(values)}行です。')
    counts = tuple(numeric_cell(value, f'{name}: {i + 1}行目')
                   for i, value in enumerate(values[:CHANNELS]))
    original_time = numeric_cell(values[1024], f'{name}: 1025行目', positive=True)
    flag = numeric_cell(values[1029], f'{name}: 1030行目')
    if flag not in (0, 1):
        raise ValueError(f'{name}: 末尾の値は0または1である必要があります。')
    dates = [str(value).strip().strip('"').strip() for value in values[1025:1029]]
    try:
        start = datetime.strptime(f'{dates[0]} {dates[1]}', '%Y-%m-%d %H:%M:%S')
        end = datetime.strptime(f'{dates[2]} {dates[3]}', '%Y-%m-%d %H:%M:%S')
    except ValueError as exc:
        raise ValueError(f'{name}: 開始・終了日時を読み取れません。') from exc
    duration = int((end - start).total_seconds())
    if duration <= 0:
        raise ValueError(f'{name}: 終了日時は開始日時より後にしてください。')
    return Spectrum(str(name), counts, start, end, duration, original_time, int(flag))


def normalized_settings(record):
    data_type = record.get('type')
    if data_type not in SETTING_FIELDS:
        raise ValueError(f'未対応のData Typeです: {data_type}')
    result = dict.fromkeys(('voltage', 'channel', 'material', 'angle', 'slit1', 'slit2', 'distance'))
    result['type'] = data_type
    for field in SETTING_FIELDS[data_type]:
        value = record.get(field)
        if field == 'channel' and (value is None or str(value).strip() == ''):
            # A manually read peak channel is optional in historical records.
            continue
        if field == 'material':
            value = str(value).strip()
            if value not in ('Au', 'Al'):
                raise ValueError('MaterialはAuまたはAlが必要です。')
        else:
            try:
                value = float(value)
            except (ValueError, TypeError) as exc:
                raise ValueError(f'{field}: 数値の測定条件が必要です。') from exc
            if not math.isfinite(value):
                raise ValueError(f'{field}: 有限の数値が必要です。')
            if field in ('voltage', 'slit1', 'slit2', 'distance') and value <= 0:
                raise ValueError(f'{field}: 正の数値が必要です。')
            if field == 'channel' and not 0 <= value < CHANNELS:
                raise ValueError('Channelは0以上1024未満が必要です。')
        result[field] = value
    return result


def json_number(number):
    if number == number.to_integral_value():
        result = int(number)
    else:
        result = float(number)
    try:
        finite = math.isfinite(result)
    except OverflowError:
        finite = False
    if not finite:
        raise ValueError('合計カウントがアップロード可能な数値の範囲を超えました。')
    return result


def merge_spectra(spectra):
    if len(spectra) < 2:
        raise ValueError('ファイルを2つ以上選んでください。')
    counts = tuple(sum((s.counts[ch] for s in spectra), Decimal(0))
                   for ch in range(CHANNELS))
    start = min(s.start for s in spectra)
    duration = sum(s.duration_sec for s in spectra)
    try:
        end = start + timedelta(seconds=duration)
    except OverflowError as exc:
        raise ValueError('統合後の終了日時が扱える範囲を超えます。') from exc
    # Keep fractional minutes: rounding to an integer would lose seconds.
    minutes = Decimal(duration) / Decimal(60)
    lines = [format(count, 'f') for count in counts]
    lines += [format(minutes, 'f'), f'"{start:%Y-%m-%d}"', f'"{start:%H:%M:%S}"',
              f'"{end:%Y-%m-%d}"', f'"{end:%H:%M:%S}"', '1']
    return {
        'csv': '\n'.join(lines) + '\n', 'start': start, 'end': end,
        'time_sec': duration, 'time_min': minutes,
        'total_count': json_number(sum(counts, Decimal(0))),
    }


def prepare_merge(spectra, settings):
    settings = normalized_settings(settings)
    merged = merge_spectra(spectra)
    digest = hashlib.sha256((json.dumps(settings, sort_keys=True) + merged['csv']).encode('utf-8')).hexdigest()
    # Distinct from original names and stable when preview is clicked again.
    filename = f'merged_{merged["start"]:%Y%m%d_%H%M%S}_{merged["time_sec"]}s_{digest[:12]}.csv'
    payload = dict(settings, file_name=filename,
                   file_content=base64.b64encode(merged['csv'].encode('utf-8')).decode('ascii'),
                   time_sec=merged['time_sec'], total_count=merged['total_count'])
    return dict(merged, payload=payload, fingerprint=digest, sources=tuple(spectra))


def uploaded_entries(value):
    """Support both ipywidgets 7 (dict) and 8 (tuple, memoryview)."""
    if isinstance(value, dict):
        return [(entry.get('name', name), bytes(entry['content'])) for name, entry in value.items()]
    return [(entry['name'], bytes(entry['content'])) for entry in value]


In [2]:
# @title 2. ログイン・ローカルCSVの統合・アップロード
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output

# This notebook is independent of experiment_state in day1/day2/day4.
prime_session = {'group_id': '', 'pin': '', 'pending': None}
uploaded_merges = set()
updating_credentials = False

input_group = widgets.Text(description='ID:', placeholder='day1・day2のGroup ID')
input_pin = widgets.Password(description='PIN:')
btn_link = widgets.Button(description='ログイン / 登録', button_style='success')
btn_logout = widgets.Button(description='ログアウト', disabled=True)
login_status = widgets.Label(value='未ログイン')
out_auth = widgets.Output()

file_uploader = widgets.FileUpload(accept='.csv', multiple=True, description='CSVを複数選択')
drop_type = widgets.Dropdown(options=list(SETTING_FIELDS), value='Measurement', description='Data Type:')
input_voltage = widgets.FloatText(value=200.0, description='Voltage (mV):')
input_channel = widgets.FloatText(value=0.0, description='Channel (ch):')
drop_material = widgets.Dropdown(options=['Au', 'Al'], description='Material:')
input_angle = widgets.FloatText(value=30.0, description='Angle (deg):')
input_slit1 = widgets.Dropdown(options=[2.0, 4.0, 6.0], value=6.0, description='Slit 1 (mm):')
input_slit2 = widgets.Dropdown(options=[2.0, 4.0, 6.0], value=6.0, description='Slit 2 (mm):')
input_distance = widgets.FloatText(value=70.0, description='Dist (mm):')
parameter_widgets = dict(voltage=input_voltage, channel=input_channel, material=drop_material,
                         angle=input_angle, slit1=input_slit1, slit2=input_slit2, distance=input_distance)
same_settings_check = widgets.Checkbox(value=False, description='選択した全CSVを同じ設定で測定したことを確認しました',
                                     layout=widgets.Layout(width='95%'))
btn_preview = widgets.Button(description='統合内容を確認', button_style='info', disabled=True)
btn_upload = widgets.Button(description='統合結果をアップロード', button_style='primary', disabled=True,
                            layout=widgets.Layout(width='230px'))
out_merge = widgets.Output()


def invalidate_preview(*args):
    prime_session['pending'] = None
    btn_upload.disabled = True
    out_merge.clear_output()


def reset_login():
    prime_session.update(group_id='', pin='', pending=None)
    login_status.value = '未ログイン'
    btn_preview.disabled = True
    btn_logout.disabled = True
    invalidate_preview()


def credentials_changed(change):
    if not updating_credentials:
        reset_login()


def api_call(action, *, group_id=None, pin=None, **data):
    account = prime_session['group_id'] if group_id is None else group_id
    secret = prime_session['pin'] if pin is None else pin
    if not account or not secret:
        raise ValueError('先にIDとPINでログインしてください。')
    response = requests.post(GAS_URL, json=dict(data, action=action, group_id=account, pin=secret),
                             timeout=(10, 60))
    response.raise_for_status()
    result = response.json()
    if not isinstance(result, dict) or result.get('status') != 'success':
        message = result.get('message', 'サーバーの応答を確認できません。') if isinstance(result, dict) else '応答形式が不正です。'
        raise ValueError(message)
    return result


def handle_link_account(button):
    global updating_credentials
    account, pin = input_group.value.strip(), input_pin.value.strip()
    reset_login()
    btn_link.disabled = True
    with out_auth:
        clear_output()
        try:
            api_call('register_group', group_id=account, pin=pin)
            prime_session.update(group_id=account, pin=pin)
            login_status.value = f'ログイン中: {account}'
            btn_preview.disabled = False
            btn_logout.disabled = False
            print(f'{account} にログインしました。ローカルCSVを2つ以上選択してください。')
        except (requests.RequestException, ValueError, TypeError, KeyError) as exc:
            reset_login()
            print(f'[ERROR] ログインに失敗しました: {exc}')
        finally:
            updating_credentials = True
            try:
                input_pin.value = ''
            finally:
                updating_credentials = False
                btn_link.disabled = False


def handle_logout(button):
    reset_login()
    input_pin.value = ''
    with out_auth:
        clear_output()
        print('ログアウトしました。')


def type_changed(*args):
    active_fields = SETTING_FIELDS[drop_type.value]
    for field, control in parameter_widgets.items():
        control.layout.display = 'flex' if field in active_fields else 'none'
    invalidate_preview()


def files_changed(*args):
    same_settings_check.value = False
    invalidate_preview()


def current_merge():
    if not prime_session['group_id']:
        raise ValueError('先にログインしてください。')
    entries = uploaded_entries(file_uploader.value)
    if len(entries) < 2:
        raise ValueError('同じ設定で測定したCSVを2つ以上選んでください。')
    if not same_settings_check.value:
        raise ValueError('全CSVが同じ測定条件であることを確認してチェックしてください。')
    settings = normalized_settings(dict(type=drop_type.value,
                                       **{key: control.value for key, control in parameter_widgets.items()}))
    spectra = [parse_spectrum(content, name) for name, content in entries]
    return prepare_merge(spectra, settings)


def handle_preview(button):
    invalidate_preview()
    with out_merge:
        try:
            pending = current_merge()
            print(f'保存先アカウント: {prime_session["group_id"]}')
            print('共通の測定条件:', {key: value for key, value in pending['payload'].items()
                                        if key in ('type', *parameter_widgets) and value is not None})
            for spectrum in pending['sources']:
                print(f'  {spectrum.name}: {spectrum.start} → {spectrum.end} / {spectrum.duration_sec}秒')
            print(f'統合後: {len(pending["sources"])}ファイル / {pending["time_sec"]}秒 ({pending["time_min"]}分)')
            print(f'開始: {pending["start"]} / 調整後の終了: {pending["end"]}')
            print(f'1024chの総カウント: {pending["total_count"]}')
            print('1025行目は合計秒数÷60、1030行目は1です。測定間の待ち時間は合計に含めません。')
            print(f'作成ファイル: {pending["payload"]["file_name"]}')
            key = (prime_session['group_id'], pending['fingerprint'])
            if key in uploaded_merges:
                print('この統合結果は、この実行中に既に送信済みです。')
                return
            prime_session['pending'] = pending
            btn_upload.disabled = False
        except (ValueError, TypeError, KeyError, IndexError, OverflowError) as exc:
            print(f'[ERROR] 統合できません: {exc}')


def handle_upload(button):
    pending = prime_session['pending']
    btn_upload.disabled = True
    with out_merge:
        try:
            if pending is None:
                raise ValueError('先に「統合内容を確認」を押してください。')
            # Revalidate the current account, files and settings before sending.
            current = current_merge()
            if current['fingerprint'] != pending['fingerprint']:
                raise ValueError('ファイルまたは条件が変わりました。統合内容を確認し直してください。')
            key = (prime_session['group_id'], pending['fingerprint'])
            if key in uploaded_merges:
                raise ValueError('この統合結果は既に送信済みです。')
            api_call('upload_data', **pending['payload'])
            uploaded_merges.add(key)
            prime_session['pending'] = None
            print(f'送信完了（サーバー受付済み）: {prime_session["group_id"]} / {pending["payload"]["file_name"]}')
        except (requests.RequestException, ValueError, TypeError, KeyError, IndexError, OverflowError) as exc:
            # Never retry a POST automatically: a timed-out request may have been saved.
            print(f'[ERROR] アップロードを確認できません: {exc}')
            print('通信エラーの場合はスプレッドシートで保存状況を確認し、必要な場合だけ再送してください。')


input_group.observe(credentials_changed, names='value')
input_pin.observe(credentials_changed, names='value')
btn_link.on_click(handle_link_account)
btn_logout.on_click(handle_logout)
btn_preview.on_click(handle_preview)
btn_upload.on_click(handle_upload)
drop_type.observe(type_changed, names='value')
file_uploader.observe(files_changed, names='value')
same_settings_check.observe(invalidate_preview, names='value')
for control in parameter_widgets.values():
    control.observe(invalidate_preview, names='value')
type_changed()
reset_login()
display(widgets.VBox([
    widgets.HTML('<h3>day4-prime — ローカルCSVの統合</h3>'),
    widgets.HBox([input_group, input_pin]), widgets.HBox([btn_link, btn_logout]), login_status, out_auth,
    widgets.HTML('同じ設定で測定したCSVを2つ以上、一度に選択してください。'), file_uploader,
    widgets.HTML('CSV内には材質・角度などの設定がありません。全ファイル共通の条件を入力してください。'),
    drop_type, *parameter_widgets.values(), same_settings_check,
    widgets.HBox([btn_preview, btn_upload]), out_merge,
]))
